# ForgeGuard: Comparative Evaluation of CNN Architectures in Detecting Digital Receipt Forgery
### Notre Dame of Midsayap College (NDMC) — BSCS Thesis Research
**Researchers**: Rogie P. Bacanto, Daniela S. Ungab  
**Adviser**: Ms. Doris Ann Mariano  

This notebook trains and benchmarks three Convolutional Neural Network (CNN) architectures on labeled mobile wallet transaction receipts (GCash) preprocessed using **Error Level Analysis (ELA 90Q / 15x)**.

All three architectures (**Basic CNN**, **MobileNetV2**, and **ResNet50**) are trained sequentially with **balanced class weighting** to handle dataset distribution.

In [ ]:
# Step 1: Clone repository if not present and navigate to thesis-system
import os
if not os.path.exists('/content/NDMC-BSCS-THESIS-PREP'):
    !git clone https://github.com/DeathKnell837/NDMC-BSCS-THESIS-PREP.git
%cd /content/NDMC-BSCS-THESIS-PREP/thesis-system
!pip install -q Pillow numpy scipy scikit-learn matplotlib seaborn


In [ ]:
# Step 2: Import libraries and verify GPU
import os, glob, time, json
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Step 3: Load and Preprocess all Dataset Samples with ELA
from preprocessing.ela import compute_ela

IMG_SIZE = (128, 128)
IMAGE_EXTENSIONS = ('*.jpg', '*.jpeg', '*.png', '*.webp')

auth_dir = 'dataset/authentic/compressed'
forged_dir = 'dataset/forged/compressed'

X, y = [], []

# Load Authentic
auth_files = []
for ext in IMAGE_EXTENSIONS:
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext)))
    auth_files.extend(glob.glob(os.path.join(auth_dir, ext.upper())))
auth_files = sorted(list(set(auth_files)))
print(f"Loading {len(auth_files)} Authentic samples...")
for f in auth_files:
    try:
        with Image.open(f) as img:
            ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
            X.append(np.array(ela, dtype=np.float32) / 255.0)
            y.append(0)
    except Exception as e:
        print(f"Skipping {f}: {e}")

# Load Forged (including AI diffusion fakes)
forged_files = []
for ext in IMAGE_EXTENSIONS:
    forged_files.extend(glob.glob(os.path.join(forged_dir, '**', ext), recursive=True))
    forged_files.extend(glob.glob(os.path.join(forged_dir, '**', ext.upper()), recursive=True))
forged_files = sorted(list(set(forged_files)))
print(f"Loading {len(forged_files)} Forged samples...")
for f in forged_files:
    try:
        with Image.open(f) as img:
            ela = compute_ela(img, quality=90, scale=15.0).resize(IMG_SIZE)
            X.append(np.array(ela, dtype=np.float32) / 255.0)
            y.append(1)
    except Exception as e:
        print(f"Skipping {f}: {e}")

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

# Stratified Split: 70% Train, 15% Val, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Dataset Prepared: Total={len(X)} | Authentic={int(np.sum(y==0))} | Forged={int(np.sum(y==1))}")
print(f"Splits: Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")


In [ ]:
# Step 4: Define 3 CNN Architectures
def build_basic_cnn():
    model = models.Sequential([
        layers.Input(shape=(128, 128, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_mobilenetv2():
    base = applications.MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_resnet50():
    base = applications.ResNet50(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = True
    # Freeze early layers, unfreeze final residual block (conv5_block3) to learn ELA noise gradients
    for layer in base.layers[:-15]:
        layer.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(2e-5), loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# Step 5: Train All 3 Architectures with Balanced Class Weights
os.makedirs('models', exist_ok=True)

n_auth = max(1, int(np.sum(y_train == 0)))
n_forg = max(1, int(np.sum(y_train == 1)))
total_train = len(y_train)
class_weights = {
    0: float(total_train / (2.0 * n_auth)),
    1: float(total_train / (2.0 * n_forg))
}
print(f"Balanced Class Weights -> Authentic (0): {class_weights[0]:.2f}, Forged (1): {class_weights[1]:.2f}")

models_dict = {
    'Basic_CNN': build_basic_cnn(),
    'MobileNetV2': build_mobilenetv2(),
    'ResNet50': build_resnet50()
}

results = {}
for name, model in models_dict.items():
    print(f"\n{'='*20} Training {name} {'='*20}")
    t0 = time.time()
    epochs = 25 if name == 'ResNet50' else 20
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=16,
        class_weight=class_weights,
        verbose=1
    )
    train_time = time.time() - t0
    
    # Test set inference
    t_inf = time.time()
    y_prob = model.predict(X_test)
    lat_ms = ((time.time() - t_inf) / len(X_test)) * 1000.0
    y_pred = (y_prob >= 0.5).astype(int).flatten()
    
    file_key = 'basic_cnn' if name == 'Basic_CNN' else name.lower()
    
    results[name] = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_test, y_pred, zero_division=0)),
        'latency_ms': float(lat_ms),
        'train_duration_s': float(train_time)
    }
    
    save_path = f"models/{file_key}.keras"
    model.save(save_path)
    print(f"Saved {save_path} | Test Accuracy: {results[name]['accuracy']*100:.2f}%")

with open('models/evaluation_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n=== Comparative Architecture Results ===")
for k, v in results.items():
    print(f"{k:14s} | Acc: {v['accuracy']*100:.2f}% | Prec: {v['precision']*100:.2f}% | Rec: {v['recall']*100:.2f}% | F1: {v['f1_score']:.4f} | Latency: {v['latency_ms']:.2f}ms")


In [ ]:
# Step 6: Package trained models and download to local machine
!zip -j /content/trained_models.zip models/basic_cnn.keras models/mobilenetv2.keras models/resnet50.keras models/evaluation_metrics.json
from google.colab import files
print("Triggering automatic browser download for trained_models.zip...")
files.download('/content/trained_models.zip')
